# [M2] DẠY MÁY HỌC TỪ DỮ LIỆU
## Buổi S02: Baseline Models & Hồi Quy Tuyến Tính (Linear Regression)

---

### Metadata Khóa Học
* **Module:** M2 — DẠY MÁY HỌC TỪ DỮ LIỆU
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook xây dựng Baseline và huấn luyện Linear Regression
* **Mức độ liên quan VOAI / Olympic AI:** Xây dựng mốc quy chiếu điểm số đầu tiên (First Working Submission) trong phòng thi.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu tại sao việc thiết lập một Baseline Model đơn giản là bước bắt buộc đầu tiên trong mọi dự án AI.
- Nắm vững công thức toán học và trực giác hình học của Hồi quy Tuyến tính (Linear Regression): y_hat = w^T * x + b.
- Hiểu trực quan Hàm mất mát Sai số Bình phương Trung bình (Mean Squared Error - MSE) và phần dư (Residuals).
- Thực thi thành thạo chu trình API chuẩn mực của Scikit-Learn: Khởi tạo -> `fit()` -> `predict()`.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M2_S01 (Feature, Target, Train/Test Split).
- Biết phương trình đường thẳng bậc nhất y = ax + b trong hình học giải tích.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Khi bạn xây dựng một mô hình AI và đạt được sai số RMSE là 15.2, con số đó là TỐT hay XẤU? Bạn hoàn toàn không thể trả lời nếu không có một **Cột mốc so sánh (Baseline)**! Nếu một thuật toán ngây thơ chỉ đơn giản đoán giá trị trung bình (Mean) mà đã đạt sai số 15.5, thì mô hình phức tạp của bạn thực chất chẳng học được điều gì có giá trị.

Hồi quy Tuyến tính (Linear Regression) là nền tảng của toàn bộ ngành khoa học dữ liệu và học máy. Mọi khái niệm tối thượng sau này — từ Trọng số (Weights), Độ lệch (Bias), Hàm mất mát (Loss Function), đến Đạo hàm suy giảm (Gradient Descent) — đều bắt nguồn từ thuật toán thanh lịch này.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Tư duy Baseline Model trong phòng thi
* **Baseline là gì?** Là mô hình đơn giản nhất có thể chạy được (Naive/Dummy Model), ví dụ: luôn đoán giá trị trung bình $\bar{y}$ (bài toán hồi quy) hoặc luôn đoán lớp chiếm đa số (bài toán phân loại).
* **Mục đích:** Thiết lập "mặt sàn" hiệu năng (Lower Bound). Bất kỳ mô hình Machine Learning thực thụ nào cũng bắt buộc phải đánh bại Baseline.

### 2. Bản chất Toán học của Hồi quy Tuyến tính (Linear Regression)
* **Phương trình dự đoán:**
  $$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_D x_D + b = \mathbf{w}^T \mathbf{x} + b$$
  Trong đó: $\mathbf{w}$ là vector trọng số (Weights/Coefficients), $b$ là hệ số tự do (Bias/Intercept).
* **Hàm mất mát (Loss Function - MSE):** Đo lường tổng bình phương khoảng cách sai lệch giữa giá trị thực tế $y_i$ và giá trị dự đoán $\hat{y}_i$:
  $$\mathcal{L}_{MSE}(\mathbf{w}, b) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$$
* **Mục tiêu của quá trình `fit()`:** Tìm bộ tham số $(\mathbf{w}^*, b^*)$ sao cho hàm mất mát $\mathcal{L}_{MSE}$ đạt giá trị cực tiểu toàn cục!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Trực quan hóa Hồi quy Tuyến tính & Phần dư (Residuals):
   y (Giá)
   ▲             • (Mẫu thực tế y_i)
   │            /│
   │           / ┆ <── Sai số phần dư e_i = y_i - y_hat_i
   │          /  ▼
   │         /───• (Dự đoán y_hat_i nằm trên đường thẳng)
   │        /
   │       /  Đường hồi quy: y_hat = w * x + b
   │      /
   └─────┴────────────────────────► x (Diện tích)
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Khoi tao du lieu mau mo phong quan he giua Kinh nghiem lam viec va Muc luong
np.random.seed(42)
n_samples = 80

experience = np.random.uniform(1, 10, size=(n_samples, 1)) # Feature X (2D array)
# Luong thuc te = 15 trieu co ban + 4.5 trieu/nam kinh nghiem + nhieu ngau nhien
salary = 15.0 + 4.5 * experience.ravel() + np.random.normal(0, 3.0, size=n_samples) # Target y (1D)

# 2. Xay dung Baseline Model (Doan gia tri trung binh cua tap Train)
dummy = DummyRegressor(strategy='mean')
dummy.fit(experience, salary)
dummy_preds = dummy.predict(experience)
dummy_mse = mean_squared_error(salary, dummy_preds)

# 3. Huan luyen mo hinh Hoi quy Tuyen tinh thuc thu (Linear Regression)
lin_reg = LinearRegression()
lin_reg.fit(experience, salary)
lin_preds = lin_reg.predict(experience)
lin_mse = mean_squared_error(salary, lin_preds)

print("===== SO SANH HIEU NANG MO HINH =====")
print(f"Baseline (Dummy Mean) MSE : {dummy_mse:.2f}")
print(f"Linear Regression MSE     : {lin_mse:.2f}")
print(f"-> Linear Regression giam sai so: {((dummy_mse - lin_mse) / dummy_mse) * 100:.1f}%")

print(f"\nTrong so hoc duoc (w): {lin_reg.coef_[0]:.2f} (Gia tri that: 4.50)")
print(f"He so bias hoc duoc (b) : {lin_reg.intercept_:.2f} (Gia tri that: 15.00)")

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa chu trình chuẩn mực từ Baseline đến Model:
- Khởi tạo bài toán ước lượng lương dựa trên năm kinh nghiệm.
- `DummyRegressor(strategy='mean')`: Thiết lập chuẩn đối sánh cơ sở.
- `LinearRegression()`: Tối ưu hóa trọng số $w$ và bias $b$ qua phương thức `fit()`.
- Chứng minh mô hình tuyến tính giảm sai số vượt trội so với Baseline ngây thơ.


In [ ]:
# 4. Truc quan hoa duong hoi quy va sai so phan du (Residuals)
plt.figure(figsize=(9, 5))
plt.scatter(experience, salary, color='#1f77b4', label='Du lieu thuc te (Data points)', alpha=0.7)
plt.plot(experience, dummy_preds, color='gray', linestyle='--', label=f'Baseline Mean (MSE={dummy_mse:.1f})')
plt.plot(experience, lin_preds, color='#d62728', linewidth=2, label=f'Linear Regression (MSE={lin_mse:.1f})')

# Ve cac duong phan du cho 5 diem dau tien
for i in range(5):
    plt.vlines(x=experience[i], ymin=min(salary[i], lin_preds[i]), ymax=max(salary[i], lin_preds[i]), 
               color='orange', linestyle=':', alpha=0.8)

plt.title('Truc quan hoa Hoi quy Tuyen tinh: Duong hoi quy va Phan du (Residuals)', fontsize=12, fontweight='bold')
plt.xlabel('So nam kinh nghiem (Years of Experience)')
plt.ylabel('Muc luong (Trieu VND)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 🔍 Code Walkthrough (Khối 2)
Khối code trực quan hóa hình học của thuật toán:
- Đường thẳng màu xám biểu diễn Baseline ngây thơ.
- Đường thẳng màu đỏ biểu diễn nghiệm tối ưu của Linear Regression.
- Các vệt kẻ đứt màu cam minh họa trực quan phần dư $e_i = y_i - \hat{y}_i$ mà hàm mất mát MSE đang tìm cách thu nhỏ tối đa.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phân tích hệ số học được
Nhìn vào giá trị `lin_reg.coef_[0]` và `lin_reg.intercept_`, hãy giải thích ý nghĩa kinh tế thực tế của 2 con số này (ví dụ: cứ mỗi năm kinh nghiệm tăng thêm, mức lương dự kiến tăng bao nhiêu triệu?).

### Bài 2 (Easy): Tính chỉ số Sai số Tuyệt đối Trung bình (MAE)
Sử dụng hàm `mean_absolute_error` từ Scikit-Learn để tính MAE cho cả Baseline và Linear Regression. Nêu điểm khác biệt trực giác giữa MAE và MSE.

### Bài 3 (Medium): Hồi quy tuyến tính đa biến (Multiple Linear Regression)
Tạo thêm một đặc trưng mới `Certifications` (số lượng chứng chỉ, từ 0 đến 5). Huấn luyện lại mô hình hồi quy với ma trận đặc trưng 2 chiều gồm cả `Experience` và `Certifications`. So sánh MSE xem có cải thiện không.

### Bài 4 (Medium): Kiểm tra tính ngẫu nhiên của phần dư (Residual Analysis)
Vẽ biểu đồ phân tán giữa giá trị dự đoán $\hat{y}$ (trục hoành) và phần dư $e = y - \hat{y}$ (trục tung). Giải thích tại sao nếu phần dư phân bố ngẫu nhiên quanh trục 0 thì mô hình được coi là phù hợp tốt (Good fit).

### Bài 5 (Challenge): Cài đặt Linear Regression từ số 0 bằng Công thức Giải tích (Normal Equation)
Nghiệm tối ưu của Hồi quy tuyến tính được tính bằng công thức ma trận đóng (Normal Equation):
$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$
Hãy dùng NumPy (`np.linalg.inv`, `@`) để tính toán trực tiếp trọng số mà không gọi thư viện Scikit-Learn và đối chiếu kết quả.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong 30 phút đầu tiên của bất kỳ cuộc thi AI nào, việc đầu tiên của một đội tuyển chuyên nghiệp là **dựng một Baseline hoàn chỉnh có thể nộp được (Working Submission)**. Đừng bao giờ bắt đầu bằng việc tinh chỉnh một mạng Transformer khổng lồ. Một pipeline Linear Regression sạch sẽ giúp bạn kiểm tra toàn bộ luồng nộp bài, kiểm tra metric chấm điểm và đảm bảo bạn có điểm số an toàn trên bảng xếp hạng trước khi thử nghiệm các kỹ thuật phức tạp hơn.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một thí sinh huấn luyện mô hình trên dữ liệu chưa xử lý và nhận được cảnh báo lỗi `NaN` hoặc trọng số khổng lồ vô lý ($10^{15}$):

```python
# CODE CÓ LỖI ĐA CỘNG TUYẾN HOÀNG TOÀN (PERFECT MULTICOLLINEARITY):
# Thi sinh tao 2 dac trung giong het nhau ve mat toan hoc:
# Feature 1: Chieu cao tinh bang met (m)
# Feature 2: Chieu cao tinh bang centimet (cm) = 100 * Feature 1
# Khi do ma tran (X^T * X) khong kha nghich (Singular Matrix), dan toi loi so hoc!
```

**Nhiệm vụ của bạn:**
1. Giải thích hiện tượng Đa cộng tuyến hoàn hảo (Multicollinearity) làm sụp đổ phép nghịch đảo ma trận trong Linear Regression như thế nào.
2. Nêu cách phát hiện và khắc phục bằng cách loại bỏ một trong hai đặc trưng tương quan tuyệt đối ($r = 1.0$).



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Mục đích quan trọng nhất của việc xây dựng mô hình Baseline ở đầu dự án là gì?
   - A. Để nộp bài lấy điểm tuyệt đối ngay lập tức.
   - B. Để làm mốc quy chiếu tối thiểu (Benchmark), chứng minh mô hình phức tạp thực sự học được tri thức có ích.
   - C. Để thay thế hoàn toàn cho bước làm sạch dữ liệu.
   - D. Để kiểm tra tốc độ xung nhịp của vi xử lý.

2. Trong công thức Hồi quy Tuyến tính $\hat{y} = \mathbf{w}^T \mathbf{x} + b$, đại lượng $b$ (Bias/Intercept) đại diện cho điều gì?
   - A. Độ dốc của đường hồi quy.
   - B. Giá trị dự đoán của $\hat{y}$ khi tất cả các đặc trưng đầu vào $x$ đều bằng 0.
   - C. Sai số lớn nhất của mô hình.
   - D. Số lượng mẫu dữ liệu trong tập Train.

3. Hàm mất mát Sai số Bình phương Trung bình (MSE) phạt các sai số lớn như thế nào so với MAE?
   - A. Nhẹ hơn MAE.
   - B. Phạt nặng hơn rất nhiều (bình phương sai số), khiến mô hình rất nhạy cảm với các điểm ngoại lai (Outliers).
   - C. Bằng đúng MAE.
   - D. Bỏ qua hoàn toàn các sai số lớn.

4. Phương thức `.fit(X, y)` trong Scikit-Learn thực chất đang thực hiện thao tác gì?
   - A. Dự đoán nhãn cho dữ liệu mới.
   - B. Tối ưu hóa các tham số nội tại của mô hình (ví dụ: weights và bias) sao cho hàm mất mát đạt giá trị nhỏ nhất trên tập dữ liệu được cung cấp.
   - C. Xóa các hàng có dữ liệu khuyết.
   - D. Vẽ biểu đồ phân phối.

5. Nếu giá trị $R^2$ (Hệ số xác định) của mô hình hồi quy bằng $0.85$, điều đó có nghĩa là gì?
   - A. Mô hình bị sai $15\%$ trên tập Test.
   - B. $85\%$ sự biến thiên của biến mục tiêu $y$ đã được giải thích bởi các đặc trưng đầu vào $X$ trong mô hình.
   - C. Mô hình đã được huấn luyện qua 85 epochs.
   - D. Dữ liệu có 85 cột.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Tư duy bắt buộc về Baseline Model trong nghiên cứu và thi đấu AI.
  - Bản chất toán học của Hồi quy Tuyến tính và cơ chế tối ưu hàm mất mát MSE.
  - Thành thạo chu trình API của Scikit-Learn: Khởi tạo, `.fit()`, `.predict()`.
* **What I can now do:**
  - Huấn luyện mô hình hồi quy tuyến tính đầu tiên và phân tích ý nghĩa các trọng số.
  - Phân tích phần dư để đánh giá độ phù hợp của mô hình.
* **What I still need to practice:**
  - Chuyển sang bài toán phân loại và làm quen với mô hình phi tuyến: **Logistic Regression & Decision Trees** ở Buổi 3.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M2_S03**, chúng ta sẽ tiến sang phân loại nhãn: **Phân Loại Nhị Phân & Cây Quyết Định (Logistic Regression & Decision Trees)** — Khám phá hàm Sigmoid, Log-Loss, cơ chế chia nhánh theo độ tinh khiết (Gini Impurity) và trực giác về Random Forest!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M2**:
- Baseline model & Dummy regressors ✓
- Linear regression intuition & MSE loss ✓
- Scikit-Learn fit/predict workflow & Residuals analysis ✓

* **Session Deliverable:** Mã nguồn huấn luyện Linear Regression đầu tiên so sánh với Baseline
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
